# Baseline recommendation
  
The first idea is to use the most effective article as a baseline.  
The most effective means the most seen but we will also need to take into account the unread ones.  
To do so I will reward newest and article subjects.  

The only dataset I will use is the *behaviors*.  

In [1]:
import pyarrow.parquet as pq
import pandas as pd
import os

## baseline prediction

In [25]:
def init_predict(dataset):
    """
    init_predict(dataset)
    this is the learning phase

    my code needs behavior data to learn
    """
    # we compute both clicked_sum and inview_sum for each article_id
    article_ids_clicked_sum = {}
    for i in dataset['article_ids_clicked']:
        for j in i:
            article_ids_clicked_sum[j] = article_ids_clicked_sum.get(j, 0) + 1

    article_ids_inview_sum = {}
    for i in dataset['article_ids_inview']:
        for j in i:
            article_ids_inview_sum[j] = article_ids_inview_sum.get(j, 0) + 1

    # this way we can compute the click rate for each article_id and sort them
    efficiency = pd.DataFrame({'clicked': article_ids_clicked_sum, 'inview': article_ids_inview_sum})
    efficiency['article_id'] = efficiency.index
    efficiency.reset_index(drop=True, inplace=True)
    efficiency['clicked'] = efficiency['clicked'].fillna(0)

    efficiency['click_rate'] = efficiency['clicked'] / efficiency['inview']
    
    efficiency = efficiency.sort_values(by='click_rate', ascending=False)

    return efficiency
    

In [50]:
def predict(article_ids_inviews):
    """
    This is the prediction function
    takes the list of the articles_ids_inviews and returns the article_id to recommend
    """
    # We return the article with the highest click rate
    for id in efficiency['article_id']:
        if id in article_ids_inviews:
            return id
    # if the article_id is not in the list, we return the first one
    return article_ids_inviews[0]

## Window accuracy

In [51]:
# test set
path = os.path.join('datas', 'ebnerd_demo', 'validation', 'behaviors.parquet')
print(path)
behaviors_tst = pq.read_table(path).to_pandas()

datas/ebnerd_demo/validation/behaviors.parquet


In [52]:
data_start = behaviors_tst['impression_time'].min()
data_end = behaviors_tst['impression_time'].max()
data_start, data_end # the time range of the behaviors

(Timestamp('2023-05-25 07:00:15'), Timestamp('2023-06-01 06:59:33'))

In [53]:
def slide(data, window_size, slide_size):
    start = data_start
    end = start + window_size
    while end <= data_end:
        yield data[(data['impression_time'] >= start) & (data['impression_time'] < end)], start, end
        start += slide_size
        end = start + window_size

In [61]:
for window, start, end in slide(behaviors_tst, window_size=pd.Timedelta(days=3), slide_size=pd.Timedelta(days=1)):
    # we split the window into training and test set
    splitting_date = end - pd.Timedelta(days=1)
    # the first days are used for training and the last day for testing
    training = window[window['impression_time'] < splitting_date].copy()
    test = window[window['impression_time'] >= splitting_date].copy()
    # we train the model
    efficiency = init_predict(training)
    # we test the model
    test.loc[:,'recommended_article_id'] = test.loc[:,'article_ids_inview'].apply(predict)
    # we compute the accuracy
    accuracy = (test['recommended_article_id'] == test['article_id']).mean()
    print(f"accuracy for the window {start} to {end} is {accuracy}")

accuracy for the window 2023-05-25 07:00:15 to 2023-05-28 07:00:15 is 0.011139401654996817
accuracy for the window 2023-05-26 07:00:15 to 2023-05-29 07:00:15 is 0.011076398750355012
accuracy for the window 2023-05-27 07:00:15 to 2023-05-30 07:00:15 is 0.006885998469778117
accuracy for the window 2023-05-28 07:00:15 to 2023-05-31 07:00:15 is 0.005552751135790005
